# Assignment 2 Notebook 01 
## Aim: Data Retrieval, Integration, CSV Cache, and MQTT Publishing

In [1]:
# Install required packages from requirements.txt
import sys
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
requirements_path = PROJECT_ROOT / "requirements.txt"

if requirements_path.exists():
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_path)
    ])
    print("Packages installed from:", requirements_path)
else:
    raise FileNotFoundError(f"requirements.txt not found at: {requirements_path}")

Packages installed from: C:\Users\The_Won\Comp5339A2\requirements.txt


In [2]:
import os
from getpass import getpass

def load_env_file(env_path):
    env_path = Path(env_path)
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

load_env_file(PROJECT_ROOT / ".env")

if not os.environ.get("OPENELECTRICITY_API_KEY"):
    os.environ["OPENELECTRICITY_API_KEY"] = getpass("OpenElectricity API key: ")

print("Project root:", PROJECT_ROOT)
print("API key loaded:", bool(os.environ.get("OPENELECTRICITY_API_KEY")))


Project root: C:\Users\The_Won\Comp5339A2
API key loaded: True


In [3]:
from argparse import Namespace
from src.electricity_pipeline import run_once

# Required assignment interval: at least one week in May 2026, 5-minute interval.
args = Namespace(
    date_start="2026-05-01T00:00:00",
    date_end="2026-05-08T00:00:00",
    interval="5m",
    chunk_size=25,          # smaller chunks reduce large-response connection resets
    request_sleep=1.0,
    max_facilities=0,       # keep 0 for the full NEM facility set; use a small number only for debugging
)

run_once(args)

Retrieving 432 facilities from 2026-05-01T00:00:00 to 2026-05-08T00:00:00
Fetched facility chunk 1/18 (25 facilities)
Fetched facility chunk 2/18 (25 facilities)
Fetched facility chunk 3/18 (25 facilities)
Fetched facility chunk 4/18 (25 facilities)
Fetched facility chunk 5/18 (25 facilities)
Fetched facility chunk 6/18 (25 facilities)
Fetched facility chunk 7/18 (25 facilities)
Fetched facility chunk 8/18 (25 facilities)
Fetched facility chunk 9/18 (25 facilities)
Fetched facility chunk 10/18 (25 facilities)
Fetched facility chunk 11/18 (25 facilities)
Fetched facility chunk 12/18 (25 facilities)
Fetched facility chunk 13/18 (25 facilities)
Fetched facility chunk 14/18 (25 facilities)
Fetched facility chunk 15/18 (25 facilities)
Fetched facility chunk 16/18 (25 facilities)
Fetched facility chunk 17/18 (25 facilities)
Fetched facility chunk 18/18 (7 facilities)
Saved facility CSV: C:\Users\The_Won\Comp5339A2\data\processed\facility_power_emissions_5min_may2026.csv rows=710,406
Saved ma

In [4]:
import pandas as pd

stream_path = PROJECT_ROOT / "data" / "processed" / "facility_power_emissions_5min_may2026.csv"
market_path = PROJECT_ROOT / "data" / "processed" / "nem_market_price_demand_5min_may2026.csv"

stream_df = pd.read_csv(stream_path)
stream_df["timestamp"] = pd.to_datetime(stream_df["timestamp"], errors="coerce")

print("Rows:", len(stream_df))
print("Facilities:", stream_df["facility_code"].nunique())
print("Timestamps:", stream_df["timestamp"].nunique())
print("Time range:", stream_df["timestamp"].min(), "to", stream_df["timestamp"].max())
print("Missing values:")
display(stream_df.isna().sum().sort_values(ascending=False).head(12))
display(stream_df.head())

market_df = pd.read_csv(market_path)
display(market_df.head())

Rows: 710406
Facilities: 353
Timestamps: 2016
Time range: 2026-05-01 00:00:00+10:00 to 2026-05-07 23:55:00+10:00
Missing values:


reporting_entity            291082
installed_capacity_mw_a1    291082
a1_facility_name            291082
longitude_a1                291082
latitude_a1                 291082
latitude_api                     0
registered_capacity_mw           0
emissions_t_co2e                 0
power_mw                         0
longitude_api                    0
timestamp                        0
facility_code                    0
dtype: int64

,timestamp,facility_code,facility_name,a1_facility_name,a1_match_score,state,network_region,facility_type,latitude,longitude,latitude_api,longitude_api,latitude_a1,longitude_a1,power_mw,emissions_t_co2e,registered_capacity_mw,installed_capacity_mw_a1,reporting_entity
0,2026-05-01 00:00:00+10:00,ADP,Adelaide Desalination,NaN,0.454545,SA,SA1,"battery, battery_charging, battery_discharging...",-35.100751,138.483357,-35.100751,138.483357,NaN,NaN,0.0,0.0,49.69,NaN,NaN
1,2026-05-01 00:00:00+10:00,AGLHAL,Hallett,Hallett Power Station,1.000000,SA,SA1,gas_ocgt,-33.348616,138.751924,-33.349528,138.751607,-33.348616,138.751924,0.0,0.0,217.00,3.091,EnergyAustralia Holdings Limited
2,2026-05-01 00:00:00+10:00,AGLSOM,Somerton,Somerton Power Station,1.000000,VIC,VIC1,gas_ocgt,-37.631601,144.952562,-37.631692,144.952802,-37.631601,144.952562,0.0,0.0,170.00,579.132,AGL Energy Limited
3,2026-05-01 00:00:00+10:00,ALDGASF,Aldoga,NaN,0.533333,QLD,QLD1,solar_utility,-23.839544,151.084900,-23.839544,151.084900,NaN,NaN,0.0,0.0,535.21,NaN,NaN
4,2026-05-01 00:00:00+10:00,ANGASTON,Angaston,Angaston Power Station,1.000000,SA,SA1,distillate,-34.503393,139.024577,-34.503389,139.024580,-34.503393,139.024577,0.0,0.0,50.00,3.091,Snowy Hydro Limited


,timestamp,price_aud_per_mwh,demand_mw
0,2026-05-01 00:00:00+10:00,40.730,20518.64
1,2026-05-01 00:05:00+10:00,40.748,20406.63
2,2026-05-01 00:10:00+10:00,42.446,20559.21
3,2026-05-01 00:15:00+10:00,42.276,20446.57
4,2026-05-01 00:20:00+10:00,39.086,20375.80


## MQTT Publisher

The publisher sends combined power + emissions messages in chronological event-time order. It uses QoS 1, waits for publish acknowledgement, sleeps exactly 0.1 seconds after each published message, and stores a state file so only new or changed records are published on later rounds.

In [5]:
from src.mqtt_helpers import (
    BROKER, PORT, TOPIC, PUBLISH_DELAY_SECONDS, ROUND_DELAY_SECONDS,
    load_stream_csv, merge_market, publish_dataframe, run_continuous_cached_publisher
)

state_path = PROJECT_ROOT / "data" / "processed" / "mqtt_published_state.json"
stream_with_market = merge_market(load_stream_csv(stream_path), market_path)

print("Broker:", BROKER)
print("Topic:", TOPIC)
print("Publish delay:", PUBLISH_DELAY_SECONDS)
print("Round delay:", ROUND_DELAY_SECONDS)
display(stream_with_market.head())

Broker: broker.hivemq.com
Topic: comp5339/2026s1/team_xinyi/electricity/facilities
Publish delay: 0.1
Round delay: 60


,timestamp,facility_code,facility_name,a1_facility_name,a1_match_score,state,network_region,facility_type,latitude,longitude,...,longitude_api,latitude_a1,longitude_a1,power_mw,emissions_t_co2e,registered_capacity_mw,installed_capacity_mw_a1,reporting_entity,price_aud_per_mwh,demand_mw
0,2026-05-01 00:00:00+10:00,ADP,Adelaide Desalination,NaN,0.454545,SA,SA1,"battery, battery_charging, battery_discharging...",-35.100751,138.483357,...,138.483357,NaN,NaN,0.0,0.0,49.69,NaN,NaN,40.73,20518.64
1,2026-05-01 00:00:00+10:00,AGLHAL,Hallett,Hallett Power Station,1.000000,SA,SA1,gas_ocgt,-33.348616,138.751924,...,138.751607,-33.348616,138.751924,0.0,0.0,217.00,3.091,EnergyAustralia Holdings Limited,40.73,20518.64
2,2026-05-01 00:00:00+10:00,AGLSOM,Somerton,Somerton Power Station,1.000000,VIC,VIC1,gas_ocgt,-37.631601,144.952562,...,144.952802,-37.631601,144.952562,0.0,0.0,170.00,579.132,AGL Energy Limited,40.73,20518.64
3,2026-05-01 00:00:00+10:00,ALDGASF,Aldoga,NaN,0.533333,QLD,QLD1,solar_utility,-23.839544,151.084900,...,151.084900,NaN,NaN,0.0,0.0,535.21,NaN,NaN,40.73,20518.64
4,2026-05-01 00:00:00+10:00,ANGASTON,Angaston,Angaston Power Station,1.000000,SA,SA1,distillate,-34.503393,139.024577,...,139.024580,-34.503393,139.024577,0.0,0.0,50.00,3.091,Snowy Hydro Limited,40.73,20518.64


In [6]:
# Smoke test: publish a small chronological sample.
# Set max_messages=None when you want to publish the whole cached week once.
PUBLISH_SMOKE_TEST = False

if PUBLISH_SMOKE_TEST:
    sent = publish_dataframe(
        stream_with_market,
        state_path=state_path,
        only_new=True,
        max_messages=20,
    )
    print("Published messages:", sent)

### The Below chunk will run continously for publish MQTT message to support the prototype dashboard update in Notebook 02, when finished accessing the prototype dashboard this chunk can be manually interrupt.

In [7]:
# Continuous execution for Tasks 1-3.
# This cell intentionally runs forever until interrupted. It reloads the cached CSV each round,
# publishes only new or changed power/emissions values, and waits 60 seconds between rounds.
RUN_CONTINUOUS_PUBLISHER = True

if RUN_CONTINUOUS_PUBLISHER:
    run_continuous_cached_publisher(
        csv_path=stream_path,
        market_path=market_path,
        state_path=state_path,
        only_new=True,
        round_delay_seconds=60,
    )

KeyboardInterrupt: 